In [0]:
import json
import traceback
from datetime import datetime
from pyspark.sql import functions as F


In [0]:

dbutils.widgets.text("batch_control_key", "", "Batch Control Key")

BATCH_CONTROL_KEY = dbutils.widgets.get("batch_control_key").strip()
assert BATCH_CONTROL_KEY, "batch_control_key parameter is required"

print(f" Batch Control Key: {BATCH_CONTROL_KEY}")



In [0]:

# ##############################################################################
# Read gold config from control table
# ##############################################################################
config_df = (
    spark.read.table(STREAMING_METADATA_TABLE)
    .filter(
        (F.col("batch_control_key") == BATCH_CONTROL_KEY) &
        (F.col("is_active") == True)
    )
)

config_rows = config_df.collect()
assert len(config_rows) > 0, (
    f"No active config found for '{BATCH_CONTROL_KEY}' in {STREAMING_METADATA_TABLE}"
)

config = config_rows[0].asDict()

silver_table_fqn = config.get("silver_table_name")
gold_table_name = config.get("gold_table_name")
gold_custom_notebook_url = (config.get("gold_custom_notebook_url") or "").strip()
source_system = config["source_system"]

AUDIT_BATCH_KEY = f"{BATCH_CONTROL_KEY}_GOLD"

print(f" Silver Table           : {silver_table_fqn}")
print(f" Gold Table             : {gold_table_name}")
print(f" Gold Custom Notebook   : {gold_custom_notebook_url or '(not configured)'}")
print(f" Source System          : {source_system}")

if not gold_custom_notebook_url:
    print("\n \u24d8 gold_custom_notebook_url not configured. Nothing to process.")
    dbutils.notebook.exit(json.dumps({"status": "SKIPPED", "message": "No gold_custom_notebook_url configured"}))


In [0]:


# ##############################################################################
# Watermark: last successful gold run timestamp
# Passed to the custom notebook so it only processes records
# inserted into silver AFTER the last successful gold load.
#
# Lazy transforms build outside try; only the action (.collect())
# is inside try to satisfy Spark Connect SCPAP005.
# ##############################################################################
watermark_query = (
    spark.read.table(AUDIT_TABLE)
    .filter(
        (F.col("BATCH_CTRL_KEY") == AUDIT_BATCH_KEY) &
        (F.col("JOB_STATUS") == "SUCCESS")
    )
    .agg(F.max("JOB_END_TIMESTAMP").alias("last_success"))
)

try:
    watermark_ts = watermark_query.collect()[0]["last_success"]
except Exception as e:
    watermark_ts = None
    print(f" \u26a0 Watermark lookup failed (defaulting to full read): {e}")

if watermark_ts:
    print(f" Watermark             : {watermark_ts} (only records after this)")
else:
    print(f" Watermark             : (none \u2014 processing all silver records)")



In [0]:


print("-" * 60)
print(" SILVER TO GOLD PROCESSING \u2014 START")
print(f" Custom Notebook: {gold_custom_notebook_url}")
print("-" * 60)

job_start = datetime.now()
records_processed = 0
status = "SUCCESS"
error_msg = None

# ##############################################################################
# Capture max _insert_timestamp from silver data for watermark safety.
# Using max data timestamp (not datetime.now()) prevents data loss when
# upstream writes more records during processing.
# ##############################################################################
silver_ts_df = spark.read.table(silver_table_fqn)
if watermark_ts:
    silver_ts_df = silver_ts_df.filter(F.col("_insert_timestamp") > F.lit(watermark_ts))

try:
    max_silver_ts = silver_ts_df.agg(F.max("_insert_timestamp")).collect()[0][0]
except Exception as e:
    max_silver_ts = None
    print(f" \u26a0 Max silver timestamp lookup failed: {e}")

try:
    result = dbutils.notebook.run(
        gold_custom_notebook_url,
        timeout_seconds=3600,
        arguments={
            "batch_control_key": BATCH_CONTROL_KEY,
            "silver_table_name": silver_table_fqn or "",
            "gold_table_name": gold_table_name or "",
            "source_system": source_system,
            "partition_column": config.get("partition_column") or "",
            "watermark_ts": str(watermark_ts) if watermark_ts else "",
        },
    )

    job_end = datetime.now()

    try:
        result_dict = json.loads(result)
        records_processed = result_dict.get("records_processed", 0)
    except (json.JSONDecodeError, TypeError):
        print(f" \u26a0 Could not parse gold notebook result as JSON (records_processed defaults to 0): {result}")

    print(f"\n \u2713 Gold custom notebook completed in {(job_end - job_start).total_seconds():.2f}s")
    print(f" Result: {result}")

except Exception as e:
    job_end = datetime.now()
    status = "FAILED"
    error_msg = str(e)
    print(f"\n \u2717 Gold custom notebook FAILED: {error_msg}")
    traceback.print_exc()


In [0]:
# ##############################################################################
# Audit record + notebook exit
# JOB_END_TIMESTAMP stores max(_insert_timestamp) from the silver
# batch - NOT datetime.now(). Prevents data loss when upstream
# writes records between the read snapshot and job completion.
# ##############################################################################
audit_rec = build_audit_record(
    AUDIT_BATCH_KEY, source_system, BATCH_CONTROL_KEY,
    records_processed, status, job_start, job_end,
    error_message=error_msg,
    extra_job_stats={
        "silver_table": silver_table_fqn,
        "gold_table": gold_table_name,
        "gold_notebook": gold_custom_notebook_url,
        "actual_job_end": str(job_end),
    },
)

# Override JOB_END_TIMESTAMP with max data timestamp for watermark safety
if status == "SUCCESS":
    audit_rec["JOB_END_TIMESTAMP"] = max_silver_ts or watermark_ts or job_start

insert_audit_record(audit_rec, AUDIT_TABLE)

# Fail-fast: if the gold custom notebook failed, raise so the orchestrator
# stops the pipeline instead of silently continuing to the next stage.
if status == "FAILED":
    raise Exception(
        f"SILVER_GOLD failed for {BATCH_CONTROL_KEY}: {error_msg}"
    )

exit_with_reconciliation(
    BATCH_CONTROL_KEY, records_processed,
    status=status, error_message=error_msg or "",
)